In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import RandomizedSearchCV
from skorch import NeuralNetClassifier
from geoai.utils_ds.visualize_ops import VisualizeOperations
from geoai.utils_ml.model_ops import ModelOperations

vis_ops = VisualizeOperations()
model_ops = ModelOperations()

In [14]:
pd.read_csv("csv_files/X_train.csv").head()

,BLUE,GREEN,RED,NIR,SWIR
0,350.0,542.0000,323.0,3277.0000,1975.3334
1,390.0,555.0000,380.0,3016.6667,1991.0000
2,1177.0,1224.6666,1268.0,1385.0000,1687.4000
3,363.2,546.5000,395.0,3244.5000,2052.0000
4,1095.6,1107.3334,1154.0,1228.0000,1573.0000


In [2]:
# load csv and convert to numpy arrays since PyTorch works with numpy arrays
X_train = pd.read_csv("csv_files/X_train.csv").values.astype(np.float32)  
X_test = pd.read_csv("csv_files/X_test.csv").values.astype(np.float32)  
y_train = pd.read_csv("csv_files/y_train_encoded.csv").values.astype(np.int64).ravel()
y_test = pd.read_csv("csv_files/y_test_encoded.csv").values.astype(np.int64).ravel()


input_dim = X_train.shape[1]
output_dim = len(np.unique(y_train))
print(f"input_dim/input features: {input_dim}, output_dim/number of classes: {output_dim}")

input_dim/input features: 5, output_dim/number of classes: 4


### NNClassifier Architecture

- **Input Layer**: `input_dim` neurons
- **Hidden Layer 1**: 
  - 16 neurons 
  - ReLU activation
- **Hidden Layer 2**: 
  - 8 neurons 
  - ReLU activation
- **Output Layer**: 
  - `output_dim` neurons (number of classes)


#### What Does a Neuron Do?
Receives Inputs → Processes Them → Applies Activation Function → Produces Output.



<img src="figures/fcnn_arch.jpg" alt="Alt text" width="700" height="500">


#### Create a Neural Network architecture: https://alexlenail.me/NN-SVG/index.html

In [3]:
# Define the architecture of the neural network
class NNClassifier(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(NNClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, 16)  # input_dim is the number of input features
        self.fc2 = nn.Linear(16, 8)  # 16 neurons to 8 neurons
        self.fc3 = nn.Linear(8, output_dim) # 8 neurons to 4 neurons since we have 4 classes


    def forward(self, x):
        x = torch.relu(self.fc1(x)) # Apply ReLU activation function to the output of the first layer
        x = torch.relu(self.fc2(x)) # Apply ReLU activation function to the output of the second layer
        x = self.fc3(x) # Apply the output layer
        return x  # Return the output tensor
    
    # Note that we dont need to define the backward pass as Skorch
    # automatically computes the gradients for us
    

### Why Neurons are Halved in Neural Network Layers

1. **Dimensionality Reduction**  
   - Encourages learning abstract, compressed features.
   - Creates a bottleneck to focus on the most relevant information.

2. **Computational Efficiency**  
   - Fewer neurons = fewer parameters, reducing overfitting risk.
   - Balances network complexity and speeds up computation.

3. **Common practice**  
   - Common in successful architectures works well for many tasks.


5. **Symmetry and Simplicity**  
   - Powers of 2 (e.g., 64, 32, 16)


In [4]:
# Define the neural network classifier
net = NeuralNetClassifier(
    NNClassifier(input_dim, output_dim),
    criterion=nn.CrossEntropyLoss, # Standard loss function for multi-class classification task
    optimizer=optim.Adam, # Adapts the learning rate for each parameter, leading to faster convergence and better performance
    max_epochs=100,  # Static number of epochs
    batch_size=2,  # Static batch size
)

In [5]:
# Define the pipeline
scaler = MinMaxScaler()
pipe = Pipeline([
    ('scaler', scaler),
    ('net', net),
])

In [6]:
# Hyperparameters for grid search
param_grid = {
    'lr': [0.01, 0.001]
}

# Perform randomized grid search
random_search = RandomizedSearchCV(net, param_grid, n_iter=10, cv=2, verbose=2)
random_search.fit(X_train, y_train)

Fitting 2 folds for each of 2 candidates, totalling 4 fits


d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\model_selection\_search.py:320: UserWarning: The total space of parameters 2 is smaller than n_iter=10. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        2.9733       0.4225        1.2842  0.7969
      2        1.2865       0.4225        1.2858  0.7659
      3        1.2877       0.4225        1.2861  0.7451
      4        1.2879       0.4225        1.2862  0.7799
      5        1.2879       0.4225        1.2862  0.7614
      6        1.2879       0.4225        1.2862  0.7474
      7        1.2879       0.4225        1.2862  0.7203
      8        1.2879       0.4225        1.2862  0.7459
      9        1.2879       0.4225        1.2862  0.7500
     10        1.2879       0.4225        1.2862  0.7377
     11        1.2879       0.4225        1.2862  0.7340
     12        1.2879       0.4225        1.2862  0.7537
     13        1.2879       0.4225        1.2862  0.7798
     14        1.2879       0.4225        1.2862  0.7430
     15        1.2879       0.4225        1.2862  0.6597
     16        1.2879       0.4

RandomizedSearchCV(cv=2,
                   estimator=<class 'skorch.classifier.NeuralNetClassifier'>[uninitialized](
  module=NNClassifier(
    (fc1): Linear(in_features=5, out_features=16, bias=True)
    (fc2): Linear(in_features=16, out_features=8, bias=True)
    (fc3): Linear(in_features=8, out_features=4, bias=True)
  ),
),
                   param_distributions={'lr': [0.01, 0.001]}, verbose=2)

In [12]:
# Get the best model
best_model = random_search.best_estimator_

print(f"Best model during training: {random_search.best_score_}")

# Save the model's state_dict
torch.save(best_model.module_.state_dict(), 'trained_models/best_model_state_dict.pth')

Best model during training: 0.9265021459227467


In [8]:
# make predictions for train and test data
fcnn_preds_train = best_model.predict(X_train)
fcnn_preds_test = best_model.predict(X_test)

# compute for the train and test data the accuracy of the simple model
print("Simple Model Accuracy")
print(f"Train Accuracy: {model_ops.calculate_classification_accuracy(y_train, fcnn_preds_train)}")
print(f"Test Accuracy: {model_ops.calculate_classification_accuracy(y_test, fcnn_preds_test)}")


Simple Model Accuracy
Train Accuracy: (0.9281115879828327, 0.9318102598743176, 0.9281115879828327, 0.9285756995110137)
Test Accuracy: (0.9399141630901288, 0.9418146417432623, 0.9399141630901288, 0.9402527998419917)


In [13]:
# Evaluate on the test set
y_pred = best_model.predict(X_test)
y_pred

array([0, 0, 2, 2, 0, 2, 1, 0, 2, 0, 0, 2, 0, 1, 2, 0, 2, 0, 1, 3, 0, 1,
       1, 1, 1, 0, 2, 2, 2, 0, 2, 1, 0, 2, 0, 2, 2, 0, 0, 0, 3, 1, 2, 0,
       2, 2, 1, 2, 2, 0, 0, 0, 3, 2, 2, 0, 3, 1, 1, 1, 0, 2, 0, 3, 2, 3,
       2, 2, 3, 1, 3, 2, 0, 2, 3, 2, 0, 0, 0, 1, 2, 0, 2, 0, 0, 1, 2, 0,
       0, 2, 2, 3, 2, 3, 0, 0, 0, 2, 2, 3, 0, 0, 3, 1, 1, 0, 0, 0, 3, 0,
       2, 1, 2, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 3, 2, 1, 1, 1, 2, 2,
       0, 1, 0, 1, 0, 3, 0, 3, 2, 3, 1, 0, 0, 2, 2, 0, 2, 2, 0, 2, 2, 2,
       0, 0, 0, 3, 1, 3, 0, 2, 1, 1, 3, 2, 1, 0, 0, 0, 2, 0, 2, 0, 0, 3,
       3, 1, 0, 1, 3, 2, 1, 0, 0, 2, 2, 0, 3, 2, 0, 3, 0, 0, 2, 0, 0, 0,
       1, 3, 0, 2, 0, 0, 3, 1, 2, 3, 0, 2, 1, 3, 3, 0, 1, 0, 0, 3, 0, 0,
       0, 2, 2, 0, 1, 1, 3, 0, 0, 3, 1, 1, 0, 2, 1, 2, 0, 3, 0, 0, 0, 2,
       2, 2, 0, 1, 0, 3, 0, 3, 0, 3, 2, 2, 2, 0, 2, 2, 3, 0, 0, 0, 1, 2,
       2, 1, 2, 3, 2, 2, 2, 2, 0, 0, 2, 2, 1, 0, 2, 1, 0, 3, 0, 0, 2, 0,
       0, 0, 3, 1, 1, 0, 2, 0, 0, 0, 2, 1, 0, 2, 3,

END